# Lab 2 : Answer from the HR policy

*Week 3 · Utrains LLMOps 8 Week Course*

Run each cell from the top. Read the printed output before you run the next cell.


## What we are achieving in this lab

Lab 1 ended at **retrieval**. You loaded `hr_policy.txt`, cut it into chunks, embedded them, stored chunk text + vector, and returned the closest chunks for a question.

Those chunks are not yet an answer an employee can read. This lab finishes the path:

- **Augmentation** = add the retrieved chunks to the prompt, so the language model can read them before it writes
- **Generation** = the language model writes the answer from that prompt

If you skip those steps, Claude has no policy text in the prompt. It will guess. That is how a wrong leave or expense answer reaches an employee.

**RAG** again, in this order:

- **Retrieval** = search your file for the pieces that match the question (Lab 1; you rebuild it here)
- **Augmentation** = add those retrieved pieces to the prompt (this lab)
- **Generation** = the language model writes the answer (this lab)

The path in code, in this order:
| Step | Plain meaning | This lab? |
|------|----------------|-----------|
| **Load** | Open `hr_policy.txt` and read it into Python as one string. | Yes (same as Lab 1) |
| **Split** | Cut that string into **chunks**. | Yes (same as Lab 1) |
| **Embed** | Turn text into a **vector** (list of numbers for meaning). | Yes (same as Lab 1) |
| **Store** | Save chunk text + vector in a vector store (the index). | Yes (same as Lab 1) |
| **Retrieve** | For a question, return the closest stored chunks. | Yes (same as Lab 1) |
| **Augment** | Add those chunks to the prompt. | Yes  new |
| **Generate** | The language model writes the answer from that prompt. | Yes  new |

Two moments in time:

| When | What runs | How often |
|------|-----------|-----------|
| The HR policy file changes | Load, split, embed, store | Once, then again only when the file changes |
| An employee asks a question | Retrieve, augment, generate | Every question |

**What you will do**

1. Load both API keys (`OPENAI_API_KEY` and `ANTHROPIC_API_KEY`).
2. Rebuild the Lab 1 index on `hr_policy.txt` (split 500 / 50, embed, store).
3. **Retrieve** the closest chunks for the reimbursement question and **read them**.
4. **Augment**: put those chunks into the prompt string. Print the prompt so you see what Claude will read.
5. **Generate**: Claude writes the answer from that prompt only.
6. Ask a question the HR policy does not answer. Search still returns three chunks. The prompt must tell Claude: if it is not in the chunks, say you cannot find it.

OpenAI turns text into vectors. Claude writes the sentences. Anthropic does not offer an embedding model. That split is on purpose.

**Before you start.** Finish Lab 1. Copy `.env.example` to `.env` in this folder. Paste both keys. Full setup is in [README.md](./README.md).

**Cost.** A few embedding calls and two short Claude replies. Fractions of a cent.


### Step 1. Load the API keys

Same pattern as Week 2 and Lab 1. `load_dotenv()` reads the `.env` file in this folder.

You need two keys:

- `OPENAI_API_KEY` — used when we embed chunks and questions (Lab 1)
- `ANTHROPIC_API_KEY` — used when Claude writes the answer (this lab)

1. Copy `.env.example` to `.env` if you have not already.
2. Paste both keys into `.env`.
3. Do not commit `.env` to git. It contains secrets.

If a later cell fails with an authentication error, fix `.env` and restart the kernel.


In [1]:
from dotenv import load_dotenv

load_dotenv()  # reads .env from this folder


True

### Step 2. Build the search index (Lab 1 again)

Lab 1 Steps 1–6, in one cell so you do not need to scroll another notebook:

1. **Load** `hr_policy.txt`
2. **Split** with `RecursiveCharacterTextSplitter`  same usable size as Lab 1: **500** characters, **50** overlap (`medium`)
3. Wrap each chunk as a **`Document`** (chunk text + chunk number)
4. **Embed + store** with OpenAI `text-embedding-3-small` and `InMemoryVectorStore.from_documents`

Under the hood, `from_documents` is the same idea as Lab 1's `embed_documents`: every chunk becomes a vector, and the store keeps **chunk text + vector** together. That saved set is the **index**.




In [3]:
from langchain_core.documents import Document
from langchain_core.vectorstores import InMemoryVectorStore
from langchain_openai import OpenAIEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter

with open("hr_policy.txt", encoding="utf-8") as f:
    POLICY = f.read()

# Same chunk size as Lab 1 Step 3 (medium = 500 / 50).
medium = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50,
).split_text(POLICY)

# Document = chunk text + a small label (chunk number).
docs = []
for i, text in enumerate(medium):
    docs.append(Document(page_content=text, metadata={"chunk": i}))

# Same embedding model as Lab 1 Step 5.
EMBED_MODEL = "text-embedding-3-small"
embeddings = OpenAIEmbeddings(model=EMBED_MODEL)

# from_documents ≈ Lab 1 embed_documents on every chunk, then save text + vector.
vectorstore = InMemoryVectorStore.from_documents(docs, embedding=embeddings)

print("file         : hr_policy.txt")
print("stored items :", len(vectorstore.store))
print("embed model  :", EMBED_MODEL)
print("store        : InMemoryVectorStore (this kernel only)")
print()
print("Each stored item has text + vector. First item:")
items = list(vectorstore.store.values())
item = items[0]
print(item)


file         : hr_policy.txt
stored items : 4
embed model  : text-embedding-3-small
store        : InMemoryVectorStore (this kernel only)

Each stored item has text + vector. First item:
{'id': '34746ed1-55fd-42f7-a64a-35af5fcbea49', 'vector': [0.005207061767578125, 0.00982666015625, 0.03875732421875, 0.03021240234375, 0.060455322265625, -0.01279449462890625, 0.0216827392578125, 0.0199432373046875, 0.018646240234375, 0.040924072265625, 0.01349639892578125, -0.0236663818359375, 0.040771484375, -0.0217742919921875, 0.007518768310546875, 0.0216064453125, -0.0016803741455078125, 0.037628173828125, -0.0186614990234375, 0.047882080078125, -0.03387451171875, 0.046051025390625, -0.038238525390625, 0.07220458984375, -0.0265655517578125, 0.021026611328125, -0.009521484375, 0.0108795166015625, -0.032684326171875, -0.09765625, 0.002658843994140625, -0.02783203125, -0.0108642578125, 0.0020923614501953125, 0.0187530517578125, 0.0248870849609375, -0.01385498046875, 0.0853271484375, 0.04559326171875, 

### Step 3. Retrieve the closest chunks

Same as Lab 1 Step 7. Step 2 only stored the index. Now we search it.

When you call `retriever.invoke(question)`, the store:

1. Runs the same idea as **`embed_query(question)`**  one vector for the question (same embedding model).
2. Compares that vector to the chunk vectors saved in Step 2.
3. Returns the closest **chunks** (the text).

`k=3` means: return the **three** closest chunks. The search always returns three, even if none of them answer the question. You will see that in Step 7.

**Read the printed chunks before you generate.** If the reimbursement section is missing here, the answer in Step 5 will be wrong, even if the sentences sound confident.


In [4]:
# k=3 means: return the 3 closest chunks.
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

QUESTION = "How do I get reimbursed for a $300 train ticket?"

# Under the hood: embed_query(question), then find closest stored vectors.
hits = retriever.invoke(QUESTION)

print("Question:", QUESTION)
print()
print("Retrieved chunks (read these before Step 4):")
print()
for i, hit in enumerate(hits, start=1):
    print("--- match", i, "  chunk", hit.metadata.get("chunk"), "---")
    print(hit.page_content)
    print()


Question: How do I get reimbursed for a $300 train ticket?

Retrieved chunks (read these before Step 4):

--- match 1   chunk 2 ---
## Section 3: Reimbursements
To submit a reimbursement, log into the finance portal at finance.example.com.
Upload your receipt as a PDF. Reimbursements take 7 to 10 business days.
For travel under $500, no pre-approval is needed. Above $500 requires
your manager and finance team approval.

--- match 2   chunk 0 ---
# HR Policy

This is the official HR policy. Use it for account setup, expense reimbursements, time off, and parental leave.

## Section 1: Setting Up Your Account
To set up your account, visit the company portal at portal.example.com.
Click the "Sign Up" button. You will receive a confirmation email within
five minutes. If you do not see the email, check your spam folder.
The portal supports two factor authentication, which we strongly recommend.

--- match 3   chunk 3 ---
## Section 4: Time Off
Submit time off requests through the HR portal. 

You should see the reimbursement section: finance portal, PDF receipt, travel under $500. A $300 ticket needs that last sentence.

Those matches are still only paragraphs. They are not yet a prompt. Step 4 adds them to the prompt. Step 5 asks Claude to write from that prompt.


### Step 4. Augment — add the chunks to the prompt

**Augmentation** means: take the retrieved chunk texts and put them into the prompt Claude will read.

Claude does not see the rest of `hr_policy.txt`. Claude only sees what you put in this prompt.

In this lab:

1. Join the three chunk texts from Step 3 into one string called `context`.
2. Build the messages:
    **system**  standing rule (Week 2): answer only from the context; if the answer is not there, say you cannot find it
    **human**  the context plus the question

The next cell only builds and **prints** that prompt. It does not call Claude yet. Generation is Step 5.


In [5]:
# Augmentation: join the retrieved chunk texts into one string for the prompt.
parts = []
for hit in hits:
    parts.append(hit.page_content)
context = "\n\n---\n\n".join(parts)

SYSTEM = (
    "You answer from an HR policy excerpt. "
    "Use only facts that appear in the context. "
    "If the context does not contain the answer, say you cannot find it. "
    "Be brief."
)

messages = [
    ("system", SYSTEM),
    ("human", "Context:\n" + context + "\n\nQuestion: " + QUESTION),
]

print("--- system message (standing rule) ---")
print(SYSTEM)
print()
print("--- human message (context + question) ---")
print(messages[1][1])


--- system message (standing rule) ---
You answer from an HR policy excerpt. Use only facts that appear in the context. If the context does not contain the answer, say you cannot find it. Be brief.

--- human message (context + question) ---
Context:
## Section 3: Reimbursements
To submit a reimbursement, log into the finance portal at finance.example.com.
Upload your receipt as a PDF. Reimbursements take 7 to 10 business days.
For travel under $500, no pre-approval is needed. Above $500 requires
your manager and finance team approval.

---

# HR Policy

This is the official HR policy. Use it for account setup, expense reimbursements, time off, and parental leave.

## Section 1: Setting Up Your Account
To set up your account, visit the company portal at portal.example.com.
Click the "Sign Up" button. You will receive a confirmation email within
five minutes. If you do not see the email, check your spam folder.
The portal supports two factor authentication, which we strongly recommend.


### Step 5. Generate — Claude writes the answer

**Generation** means: send the augmented prompt from Step 4 to a language model. The model writes the sentence answer.

We use Claude `claude-haiku-4-5` through LangChain. OpenAI does not write this answer.

`temperature=0` means: pick the most likely next words, not a random creative phrasing. For a policy answer, that is what you want.

The next cell uses the same `messages` list you printed in Step 4.


In [6]:
from langchain_anthropic import ChatAnthropic

CHAT_MODEL = "claude-haiku-4-5"
llm = ChatAnthropic(model=CHAT_MODEL, temperature=0)

# Generation: send the Step 4 prompt. Claude answers from that context only.
answer = llm.invoke(messages)

print("chat model:", CHAT_MODEL)
print()
print("--- answer ---")
print(answer.content)


chat model: claude-haiku-4-5

--- answer ---
To get reimbursed for a $300 train ticket:

1. Log into the finance portal at finance.example.com
2. Upload your receipt as a PDF
3. Since $300 is under $500, no pre-approval is needed
4. Your reimbursement will be processed in 7 to 10 business days


Check two things, in this order:

1. Did search return the reimbursement chunk? (Step 3)
2. Did Claude's answer stay inside those chunks? It should mention the finance portal, a PDF receipt, and that travel under $500 does not need pre-approval.

**Grounded** means: you can point at a sentence in the retrieved text that supports the answer. Claude never saw parental leave or PTO in this prompt, so it should not mention them.


### Step 6. A question the HR policy does not answer

Ask about a pet-bereavement policy. That topic is not in `hr_policy.txt`.

Search still returns three chunks, because `k=3`. Those chunks will be the closest *wrong* topics (often time off or **PTO** — paid time off, the vacation or personal leave days an employee can use — because the question is about a kind of leave).

Do the same loop as Steps 3–5, in order:

1. **Retrieve** — print the three chunks and read them
2. **Augment** — put those chunks in the prompt
3. **Generate** — Claude should say it cannot find the answer, instead of inventing a pet policy


In [7]:
MISSING = "What is the company's pet-bereavement policy?"

# 1. Retrieve (same as Step 3)
missing_hits = retriever.invoke(MISSING)

print("Question:", MISSING)
print()
print("Retrieved chunks (closest wrong topics — read them):")
print()
for i, hit in enumerate(missing_hits, start=1):
    print("--- match", i, "  chunk", hit.metadata.get("chunk"), "---")
    print(hit.page_content)
    print()

# 2. Augment (same as Step 4)
parts = []
for hit in missing_hits:
    parts.append(hit.page_content)
missing_context = "\n\n---\n\n".join(parts)

missing_messages = [
    (
        "system",
        "You answer using ONLY the provided context. "
        "If the answer is not in the context, say you cannot find it. Do not guess.",
    ),
    (
        "human",
        "Context:\n" + missing_context + "\n\nQuestion: " + MISSING,
    ),
]

print("--- augmented human message (preview) ---")
print(missing_messages[1][1][:400], "...")
print()

# 3. Generate (same as Step 5)
refusal = llm.invoke(missing_messages)
print("--- answer ---")
print(refusal.content)


Question: What is the company's pet-bereavement policy?

Retrieved chunks (closest wrong topics — read them):

--- match 1   chunk 3 ---
## Section 4: Time Off
Submit time off requests through the HR portal. We have an unlimited PTO
policy with a 2 week minimum suggested per year. Sick days are tracked
separately. To request bereavement leave, contact HR directly.

## Section 5: Parental Leave
Full-time employees receive 16 weeks of paid parental leave. The leave
may be taken any time in the first 12 months after the child's arrival.
Notify HR at least 30 days in advance when the date is predictable.

--- match 2   chunk 0 ---
# HR Policy

This is the official HR policy. Use it for account setup, expense reimbursements, time off, and parental leave.

## Section 1: Setting Up Your Account
To set up your account, visit the company portal at portal.example.com.
Click the "Sign Up" button. You will receive a confirmation email within
five minutes. If you do not see the email, check your sp

The chunks you printed are not about pets. They are the closest leftover topics in the file.

A grounded prompt should refuse. If Claude invents a pet policy from a PTO chunk, that is a **hallucination**: a fluent answer that is not in the source. Lab 3 shows more ways that happens.

Later, production systems add a score cutoff: if nothing is close enough, **your code** says "I don't know" and does not call the model. We do not add that cutoff today.


### Optional. The same loop on a PDF

Companies often keep policies as PDFs, not `.txt` files. If you drop a short PDF next to this notebook and name it `hr_policy.pdf`, this cell reads the text layer with `pypdf`, then splits, stores, and searches the same way as Lab 1.

If you have no PDF, skip the cell. If the PDF is a scan with no text, the pages will print empty. That is a file problem, not an embeddings problem. Week 4 looks at harder PDFs.


In [ ]:
from pathlib import Path

from pypdf import PdfReader

pdf_path = Path("hr_policy.pdf")

if not pdf_path.exists():
    print("No hr_policy.pdf found. Skip this cell, or drop a PDF next to the notebook.")
else:
    pages = []
    reader = PdfReader(str(pdf_path))
    for i, page in enumerate(reader.pages):
        text = page.extract_text() or ""
        pages.append(
            Document(
                page_content=text,
                metadata={"source": str(pdf_path), "page": i + 1},
            )
        )

    # Same split idea as Lab 1 / Step 2, but starting from page Documents.
    splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
    pdf_chunks = splitter.split_documents(pages)
    pdf_store = InMemoryVectorStore.from_documents(pdf_chunks, embedding=embeddings)
    pdf_hits = pdf_store.as_retriever(search_kwargs={"k": 3}).invoke(QUESTION)

    print("pages :", len(reader.pages))
    print("chunks:", len(pdf_chunks))
    print("file  :", pdf_path)
    print()
    for hit in pdf_hits:
        preview = hit.page_content.replace("\n", " ")
        if len(preview) > 140:
            preview = preview[:140] + "..."
        print("p." + str(hit.metadata.get("page")), preview)


## What you should be able to explain

- Lab 1 finds chunks. Lab 2 **augments** the prompt with those chunks, then **generates** the answer. All three RAG letters are required, in that order.
- Always print the retrieved chunks before you trust the answer. Wrong chunks mean a wrong answer, even if the wording sounds sure.
- Augmentation is the `context` string in the prompt. Generation is `llm.invoke` on that prompt.
- Claude writes the sentences. OpenAI turns text into vectors. Anthropic does not offer an embedding model.
- Search with `k=3` always returns three chunks. The prompt must say: if the answer is not in those chunks, do not guess.

**Try it as a product.** [`handbook-chat/`](./handbook-chat/) is this same loop in a Streamlit app on the same HR policy. See the Week 3 [README.md](./README.md) to run it.

**Lab 3** uses this loop on a messy set of documents, so you can see when search plus generate is not enough.
